# 01 — Content engine architecture

*Applied Labs · Domain: Prompt Engineering · Advanced · ~35 min*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AI2026/castalia/blob/main/labs/07_content_seo_engine/01_architecture.ipynb)

**Prerequisites:** `00_first_principles.ipynb`

Now that we understand the first principles — ranking factors, search
intent, brand voice, and quality rubrics — we need an **architecture**
that orchestrates them into a coherent pipeline.

> **Learning goals**
> 1. Map the end-to-end content engine pipeline.
> 2. Build a topic research agent with structured output.
> 3. Design a brief generator that translates research into writing specs.
> 4. Integrate brand voice via RAG-style retrieval.
> 5. Implement post-writing SEO optimization.
> 6. Design the quality evaluation and revision loop.

## 0 · Setup

In [ ]:
!pip install -q "openai>=1.0.0" "sentence-transformers>=2.2.0" "textstat>=0.7.0"

In [ ]:
import os
import json
import math
import re
from typing import Any

import numpy as np
import textstat

OPENAI_API_KEY: str | None = os.getenv("OPENAI_API_KEY")
print(f"OpenAI key configured: {OPENAI_API_KEY is not None}")
print("Environment ready ✓")

## 1 · System architecture

```
┌──────────┐    ┌────────────────┐    ┌─────────────────┐    ┌──────────────┐
│  Topic   │───▶│  Topic         │───▶│  SERP           │───▶│  Brief       │
│  Input   │    │  Researcher    │    │  Analyzer       │    │  Generator   │
└──────────┘    └────────────────┘    └─────────────────┘    └──────┬───────┘
                                                                    │
                                                                    ▼
┌──────────┐    ┌────────────────┐    ┌─────────────────┐    ┌──────────────┐
│  Final   │◀───│  Quality       │◀───│  SEO            │◀───│  Article     │
│  Output  │    │  Evaluator     │    │  Optimizer      │    │  Writer      │
└──────────┘    └───────┬────────┘    └─────────────────┘    └──────────────┘
                        │
                  ┌─────▼──────┐
                  │  Revision  │  (loop if score < threshold)
                  │  Loop      │
                  └────────────┘
```

Each stage is a **self-contained agent** with defined inputs, outputs,
and a quality contract. This lets us test, improve, and swap components
independently.

In [ ]:
# Pipeline stage contract
from dataclasses import dataclass, field


@dataclass
class PipelineStage:
    """Definition of a single pipeline stage."""
    name: str
    inputs: list[str]
    outputs: list[str]
    description: str
    quality_gate: str = ""


PIPELINE_STAGES: list[PipelineStage] = [
    PipelineStage(
        name="topic_researcher",
        inputs=["keyword", "target_audience"],
        outputs=["research_brief"],
        description="Analyze keyword, mine questions, review competitors",
        quality_gate="research_depth_score >= 0.7",
    ),
    PipelineStage(
        name="serp_analyzer",
        inputs=["keyword", "research_brief"],
        outputs=["serp_analysis"],
        description="Analyze top-ranking content for intent and gaps",
        quality_gate="competitor_count >= 3",
    ),
    PipelineStage(
        name="brief_generator",
        inputs=["research_brief", "serp_analysis"],
        outputs=["content_brief"],
        description="Create structured outline with SEO requirements",
        quality_gate="outline_has_h2_and_h3",
    ),
    PipelineStage(
        name="article_writer",
        inputs=["content_brief", "brand_voice"],
        outputs=["draft_article"],
        description="Write full article following brief and voice guide",
        quality_gate="word_count >= 800",
    ),
    PipelineStage(
        name="seo_optimizer",
        inputs=["draft_article", "content_brief"],
        outputs=["optimized_article"],
        description="Check and fix keyword usage, headings, meta tags",
        quality_gate="seo_score >= 0.8",
    ),
    PipelineStage(
        name="quality_evaluator",
        inputs=["optimized_article", "brand_voice", "content_brief"],
        outputs=["quality_report", "revision_instructions"],
        description="Multi-dimensional scoring and revision guidance",
        quality_gate="total_score >= 0.75",
    ),
]

print("Content Engine Pipeline")
print("=" * 60)
for i, stage in enumerate(PIPELINE_STAGES):
    arrow = "  → " if i > 0 else "    "
    print(f"{arrow}{i+1}. {stage.name}")
    print(f"       in:  {stage.inputs}")
    print(f"       out: {stage.outputs}")
    print(f"       gate: {stage.quality_gate}")

## 2 · Topic research agent

The researcher takes a **keyword** and produces a structured brief
containing audience questions, competitor analysis, keyword suggestions,
and content gaps. In production this would call SERP APIs; here we
simulate with realistic data.

In [ ]:
# Simulated SERP and research data
SIMULATED_SERP_DATA: dict[str, dict[str, Any]] = {
    "kubernetes autoscaling": {
        "people_also_ask": [
            "How does Kubernetes horizontal pod autoscaler work?",
            "What is the difference between HPA and VPA?",
            "How to set up cluster autoscaler?",
            "What metrics should I use for autoscaling?",
            "Is Kubernetes autoscaling cost-effective?",
        ],
        "top_results": [
            {"title": "K8s Autoscaling Deep Dive", "word_count": 2200, "h2_count": 6,
             "angle": "Technical HPA tutorial"},
            {"title": "Complete Guide to K8s Scaling", "word_count": 1800, "h2_count": 5,
             "angle": "Beginner-friendly overview"},
            {"title": "Autoscaling Best Practices", "word_count": 1500, "h2_count": 4,
             "angle": "Production tips and tricks"},
        ],
        "related_keywords": [
            "horizontal pod autoscaler", "vertical pod autoscaler",
            "cluster autoscaler", "KEDA event-driven", "custom metrics",
        ],
    },
    "content marketing strategy": {
        "people_also_ask": [
            "How do I create a content marketing strategy?",
            "What are the 5 pillars of content marketing?",
            "How much does content marketing cost?",
            "What tools are best for content marketing?",
            "How to measure content marketing ROI?",
        ],
        "top_results": [
            {"title": "Content Strategy Framework 2025", "word_count": 3000, "h2_count": 8,
             "angle": "Comprehensive framework"},
            {"title": "Content Marketing for Startups", "word_count": 1600, "h2_count": 5,
             "angle": "Budget-conscious approach"},
            {"title": "B2B Content Strategy Guide", "word_count": 2400, "h2_count": 7,
             "angle": "Enterprise-focused"},
        ],
        "related_keywords": [
            "content calendar", "editorial strategy", "content pillars",
            "content distribution", "content repurposing",
        ],
    },
}


def research_topic(keyword: str) -> dict[str, Any]:
    """Perform topic research using simulated SERP data.

    Parameters
    ----------
    keyword : primary target keyword

    Returns
    -------
    dict – structured research brief.
    """
    data = SIMULATED_SERP_DATA.get(keyword, SIMULATED_SERP_DATA["kubernetes autoscaling"])

    avg_word_count = int(np.mean([r["word_count"] for r in data["top_results"]]))
    avg_h2_count = int(np.mean([r["h2_count"] for r in data["top_results"]]))
    angles_covered = [r["angle"] for r in data["top_results"]]

    gaps: list[str] = []
    if avg_word_count < 2000:
        gaps.append("Opportunity for more comprehensive coverage")
    if not any("cost" in a.lower() or "roi" in a.lower() for a in angles_covered):
        gaps.append("No competitor covers cost/ROI analysis")
    if not any("case study" in a.lower() or "example" in a.lower() for a in angles_covered):
        gaps.append("Missing real-world case studies")

    return {
        "keyword": keyword,
        "people_also_ask": data["people_also_ask"],
        "competitor_analysis": data["top_results"],
        "related_keywords": data["related_keywords"],
        "content_gaps": gaps,
        "target_word_count": max(avg_word_count + 500, 1500),
        "target_h2_count": avg_h2_count + 2,
        "research_depth_score": round(
            min(len(data["people_also_ask"]) / 5 + len(gaps) / 3, 1.0), 2
        ),
    }


# Demo
brief = research_topic("kubernetes autoscaling")
print(json.dumps(brief, indent=2))

## 3 · Brief generation

The brief converts raw research into a **writing specification**: target
keyword, intent, outline with H2/H3 headings, subtopics to cover,
competitor gaps to exploit, and a word count target.

In [ ]:
def generate_brief(research: dict[str, Any], intent: str = "informational") -> dict[str, Any]:
    """Generate a content brief from research data.

    Parameters
    ----------
    research : output of research_topic()
    intent : primary search intent type

    Returns
    -------
    dict – structured content brief.
    """
    keyword = research["keyword"]
    paa = research["people_also_ask"]
    gaps = research["content_gaps"]
    related = research["related_keywords"]

    # Build outline from PAA + gaps
    outline: list[dict[str, Any]] = [
        {"level": "h2", "text": f"What is {keyword}?",
         "subtopics": [f"Core concepts of {keyword}", "Why it matters"]},
        {"level": "h2", "text": f"How {keyword} works",
         "subtopics": [related[0] if related else "Key mechanism",
                       related[1] if len(related) > 1 else "Architecture"]},
    ]

    # Add H2s from PAA
    for q in paa[:3]:
        outline.append({
            "level": "h2",
            "text": q.rstrip("?"),
            "subtopics": [f"Deep dive: {q}"],
        })

    # Add gap-filling sections
    for gap in gaps[:2]:
        outline.append({
            "level": "h2",
            "text": gap,
            "subtopics": [f"Original analysis: {gap}"],
        })

    return {
        "target_keyword": keyword,
        "search_intent": intent,
        "outline": outline,
        "related_keywords_to_include": related,
        "competitor_gaps_to_exploit": gaps,
        "target_word_count": research["target_word_count"],
        "seo_requirements": {
            "keyword_in_title": True,
            "keyword_in_first_100_words": True,
            "keyword_in_at_least_one_h2": True,
            "meta_description_length": "150-160 chars",
            "include_internal_links": 3,
            "include_external_links": 2,
        },
    }


research = research_topic("content marketing strategy")
brief = generate_brief(research, intent="informational")
print(f"Brief for: {brief['target_keyword']}")
print(f"Intent: {brief['search_intent']}")
print(f"Target words: {brief['target_word_count']}")
print(f"\nOutline ({len(brief['outline'])} sections):")
for section in brief["outline"]:
    print(f"  {section['level'].upper()}: {section['text']}")
    for sub in section.get("subtopics", []):
        print(f"    └─ {sub}")

## 4 · Writing with brand voice

The writer receives a **brief** plus a **voice guide** and produces
the article. Brand voice is maintained through few-shot examples
injected into the system prompt — conceptually similar to RAG over
past content.

In [ ]:
# Voice-aware prompt builder
BRAND_VOICES: dict[str, dict[str, Any]] = {
    "technical_b2b": {
        "tone": ["authoritative", "precise", "data-driven"],
        "vocabulary": ["leverage", "optimize", "infrastructure", "scalable"],
        "persona": "Senior engineering lead writing for CTOs",
        "example": (
            "Horizontal pod autoscaling leverages real-time metric streams "
            "to dynamically adjust replica counts. Our benchmarks show a 37% "
            "reduction in compute waste."
        ),
    },
    "friendly_consumer": {
        "tone": ["conversational", "encouraging", "clear"],
        "vocabulary": ["easy", "simple", "step-by-step", "you'll love"],
        "persona": "Helpful friend making complex topics approachable",
        "example": (
            "Ever feel like your cloud bill is out of control? The good news "
            "is autoscaling can save you serious money — think of it like a "
            "smart thermostat for your servers!"
        ),
    },
}


def build_writer_prompt(
    brief: dict[str, Any],
    voice_name: str,
) -> dict[str, str]:
    """Build system + user prompts for the article writer.

    Parameters
    ----------
    brief : output of generate_brief()
    voice_name : key into BRAND_VOICES

    Returns
    -------
    dict with 'system' and 'user' prompt strings.
    """
    voice = BRAND_VOICES[voice_name]

    outline_str = "\n".join(
        f"  {s['level'].upper()}: {s['text']}" for s in brief["outline"]
    )

    system = (
        f"You are a professional content writer.\n"
        f"Voice: {', '.join(voice['tone'])}\n"
        f"Persona: {voice['persona']}\n"
        f"Preferred vocabulary: {', '.join(voice['vocabulary'])}\n\n"
        f"Style example:\n\"{voice['example']}\"\n\n"
        f"IMPORTANT: Maintain this voice consistently throughout the article."
    )

    user = (
        f"Write a {brief['target_word_count']}-word article about "
        f"'{brief['target_keyword']}'.\n\n"
        f"Search intent: {brief['search_intent']}\n\n"
        f"Outline:\n{outline_str}\n\n"
        f"SEO requirements:\n"
        f"- Include keyword in first 100 words\n"
        f"- Use related keywords: {', '.join(brief.get('related_keywords_to_include', [])[:5])}\n"
        f"- Target readability: Flesch score 50-70\n\n"
        f"Competitor gaps to exploit:\n"
        + "\n".join(f"- {g}" for g in brief.get("competitor_gaps_to_exploit", []))
    )

    return {"system": system, "user": user}


# Demo
prompts = build_writer_prompt(brief, "technical_b2b")
print("── SYSTEM PROMPT (first 300 chars) ──")
print(prompts["system"][:300])
print("\n── USER PROMPT (first 400 chars) ──")
print(prompts["user"][:400])

## 5 · SEO optimization

After the writer produces a draft, the **SEO optimizer** checks and
fixes: keyword density, heading structure, meta description generation,
and readability scoring.

In [ ]:
def analyze_seo(
    article_text: str,
    keyword: str,
    target_word_count: int = 1500,
) -> dict[str, Any]:
    """Analyze SEO compliance of an article.

    Parameters
    ----------
    article_text : the full article text
    keyword : target keyword
    target_word_count : expected word count

    Returns
    -------
    dict with SEO analysis results and overall score.
    """
    word_count = len(article_text.split())
    keyword_lower = keyword.lower()
    text_lower = article_text.lower()

    # Keyword analysis
    keyword_count = text_lower.count(keyword_lower)
    keyword_density = (keyword_count / word_count * 100) if word_count else 0.0

    # Heading analysis
    h2_matches = re.findall(r"^##\s+(.+)$", article_text, re.MULTILINE)
    h3_matches = re.findall(r"^###\s+(.+)$", article_text, re.MULTILINE)
    keyword_in_headings = any(keyword_lower in h.lower() for h in h2_matches)

    # First 100 words check
    first_100 = " ".join(article_text.split()[:100]).lower()
    keyword_in_intro = keyword_lower in first_100

    # Readability
    flesch_score = textstat.flesch_reading_ease(article_text)
    gunning_fog = textstat.gunning_fog(article_text)

    # Scoring
    checks = {
        "word_count_adequate": word_count >= target_word_count * 0.8,
        "keyword_density_ok": 0.5 <= keyword_density <= 2.5,
        "keyword_in_headings": keyword_in_headings,
        "keyword_in_intro": keyword_in_intro,
        "has_h2_structure": len(h2_matches) >= 3,
        "readability_ok": 40 <= flesch_score <= 80,
    }
    score = sum(checks.values()) / len(checks)

    return {
        "word_count": word_count,
        "keyword_count": keyword_count,
        "keyword_density": round(keyword_density, 2),
        "h2_headings": h2_matches,
        "h3_headings": h3_matches,
        "keyword_in_intro": keyword_in_intro,
        "flesch_reading_ease": round(flesch_score, 1),
        "gunning_fog_index": round(gunning_fog, 1),
        "checks": checks,
        "seo_score": round(score, 2),
    }


# Demo with sample article
sample_article = """## What is Kubernetes Autoscaling?

Kubernetes autoscaling is the process of automatically adjusting compute
resources based on real-time demand. Whether you're running a small
startup or a large enterprise, kubernetes autoscaling helps you optimize
costs while maintaining performance.

## How Horizontal Pod Autoscaler Works

The Horizontal Pod Autoscaler (HPA) monitors CPU utilization and memory
usage to scale the number of pod replicas up or down. This ensures your
application handles traffic spikes without manual intervention.

## Vertical Pod Autoscaler vs HPA

While HPA adjusts the number of pods, the Vertical Pod Autoscaler (VPA)
adjusts the resource requests and limits for individual containers. Each
approach has trade-offs depending on your workload characteristics.

## Best Practices for Production

Setting up kubernetes autoscaling in production requires careful tuning
of metrics, thresholds, and cooldown periods. Start with CPU-based
scaling and gradually add custom metrics from Prometheus or Datadog.
"""

seo_result = analyze_seo(sample_article, "kubernetes autoscaling")
print("SEO Analysis")
print("=" * 40)
for k, v in seo_result.items():
    if k != "checks":
        print(f"  {k:.<30} {v}")
print("\nChecks:")
for check, passed in seo_result["checks"].items():
    print(f"  {'✓' if passed else '✗'} {check}")

## 6 · Quality eval and revision loop

The evaluator scores the article on multiple dimensions and, if the
score falls below a threshold, generates **specific revision
instructions** that feed back into the writer.

In [ ]:
QUALITY_DIMENSIONS: dict[str, dict[str, Any]] = {
    "depth": {
        "weight": 0.25,
        "rubric": "Does the content go beyond surface-level coverage?",
    },
    "originality": {
        "weight": 0.20,
        "rubric": "Does it offer unique insights, data, or framing?",
    },
    "seo_compliance": {
        "weight": 0.20,
        "rubric": "Does it meet all technical SEO requirements?",
    },
    "readability": {
        "weight": 0.20,
        "rubric": "Is it clear, well-structured, and engaging?",
    },
    "voice_consistency": {
        "weight": 0.15,
        "rubric": "Does it maintain brand voice throughout?",
    },
}


def evaluate_quality(
    dimension_scores: dict[str, int],
    seo_score: float,
    dimensions: dict[str, dict[str, Any]] = QUALITY_DIMENSIONS,
) -> dict[str, Any]:
    """Evaluate overall article quality and generate revision guidance.

    Parameters
    ----------
    dimension_scores : dict mapping dimension → integer score (1-5)
    seo_score : output of analyze_seo (0-1 float)
    dimensions : the quality dimensions

    Returns
    -------
    dict with scores, pass/fail, and revision instructions.
    """
    # Override SEO from actual measurement
    dimension_scores["seo_compliance"] = min(5, max(1, round(seo_score * 5)))

    total = 0.0
    details: dict[str, Any] = {}
    revisions: list[str] = []

    for dim, meta in dimensions.items():
        raw = dimension_scores.get(dim, 3)
        weighted = (raw / 5.0) * meta["weight"]
        total += weighted
        details[dim] = {"score": raw, "weighted": round(weighted, 3)}
        if raw <= 2:
            revisions.append(f"CRITICAL: Improve {dim} — {meta['rubric']}")
        elif raw == 3:
            revisions.append(f"IMPROVE: {dim} needs strengthening")

    passed = total >= 0.75
    return {
        "dimensions": details,
        "total_score": round(total, 3),
        "passed": passed,
        "revision_instructions": revisions if not passed else [],
    }


def revision_loop(
    max_revisions: int = 3,
    initial_scores: dict[str, int] | None = None,
) -> list[dict[str, Any]]:
    """Simulate the revision loop with improving scores.

    Parameters
    ----------
    max_revisions : maximum revision iterations
    initial_scores : starting dimension scores

    Returns
    -------
    list of evaluation results per iteration.
    """
    scores = initial_scores or {"depth": 2, "originality": 2, "readability": 3, "voice_consistency": 3}
    seo = 0.5
    history: list[dict[str, Any]] = []

    for i in range(max_revisions + 1):
        result = evaluate_quality(dict(scores), seo)
        result["iteration"] = i
        history.append(result)

        if result["passed"]:
            print(f"  Iteration {i}: score={result['total_score']:.3f} ✓ PASSED")
            break
        else:
            print(f"  Iteration {i}: score={result['total_score']:.3f} ✗ revising…")
            for inst in result["revision_instructions"]:
                print(f"    → {inst}")

        # Simulate improvement
        for dim in scores:
            scores[dim] = min(5, scores[dim] + 1)
        seo = min(1.0, seo + 0.2)

    return history


print("Revision Loop Simulation")
print("=" * 40)
history = revision_loop()

## Exercises

1. **Add a SERP analyzer stage** — Extend `SIMULATED_SERP_DATA` with a
   third topic. Build a `analyze_serp()` function that computes average
   word count, heading depth, and identifies the dominant intent.
2. **Meta description generator** — Write a function that takes the
   article text and keyword, then generates a 150–160 character meta
   description. Score it for keyword inclusion and CTA presence.
3. **Voice drift detector** — Extend `evaluate_quality` to detect
   voice drift by comparing embedding similarity of each paragraph
   against the brand voice example. Flag paragraphs that drift > 0.3
   cosine distance.

## Takeaways

* The content engine has **six stages**, each with defined I/O and
  quality gates.
* Research and brief generation are the *foundation* — garbage in,
  garbage out.
* Brand voice is injected via **system prompt + few-shot examples**,
  conceptually similar to RAG.
* SEO optimization is a **post-processing step**, not a writing constraint.
* The **revision loop** is what separates "good enough" from "publish-ready".

## What's next

In **02 — Build** we'll implement the full pipeline end-to-end with
OpenAI, generate real articles in multiple brand voices, and run the
complete research → write → optimize → evaluate → revise cycle.

---

## References

1. Google — "Search Quality Evaluator Guidelines" (2024)
   https://guidelines.raterhub.com
2. Ahrefs — "How to Do Keyword Research for SEO" (2024)
   https://ahrefs.com/blog/keyword-research/
3. Nielsen Norman Group — "How People Read on the Web" (2020)
   https://www.nngroup.com/articles/how-people-read-online/